In [ ]:
!pip install evaluate
!pip install transformers datasets --upgrade


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 71.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 494.8/494.8 kB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 15.6 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
  Attempting uninstall: transformers
    Found existing installation: transformers 4.53.1
    Uninstalling transformers-4.53.1:
      Successfully uninstalled transformers-4.53.1
  Attempting uninstall: datasets
    Found existing installation: datasets 2.14.4
    Uninstalling datasets-2.14.4:
      Successfully uninstalled datasets-2.14.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are in

In [ ]:
# Step 1: Import
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import evaluate
import numpy as np

# Step 2: Load MRPC dataset
raw_datasets = load_dataset("glue", "mrpc")

# Step 3: Load tokenizer
checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

# Step 4: Tokenize the dataset
def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)

tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)

# Step 5: Create data collator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Step 6: Load pretrained BERT model for classification
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

# Step 7: Define compute_metrics function
def compute_metrics(eval_preds):
    metric = evaluate.load("glue", "mrpc")
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

# Step 8: Define training arguments with wandb disabled
training_args = TrainingArguments(
    output_dir="test-trainer",            # Where to save model
    learning_rate=2e-5,                   # Learning rate
    per_device_train_batch_size=8,       # Batch size for training
    per_device_eval_batch_size=8,        # Batch size for eval
    num_train_epochs=3,                   # Number of epochs
    weight_decay=0.01,                   # Regularization
    push_to_hub=False,                   # Don’t upload to Hugging Face hub
    report_to=[]                        # Disable wandb and other trackers
)

# Step 9: Create Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

# Step 10: Train the model
trainer.train()


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/649k [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/75.7k [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/308k [00:00<?, ?B/s]

...

Map:   0%|          | 0/1725 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-1-3379593598.py:47: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
500,0.512500
1000,0.263000


TrainOutput(global_step=1377, training_loss=0.3282512202155529, metrics={'train_runtime': 9197.2021, 'train_samples_per_second': 1.196, 'train_steps_per_second': 0.15, 'total_flos': 405114969714960.0, 'train_loss': 0.3282512202155529, 'epoch': 3.0})

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# ✅ Save trained model
model.save_pretrained("/content/drive/MyDrive/bert_mrpc_model")

# ✅ Save tokenizer
tokenizer.save_pretrained("/content/drive/MyDrive/bert_mrpc_model")


Mounted at /content/drive


('/content/drive/MyDrive/bert_mrpc_model/tokenizer_config.json',
 '/content/drive/MyDrive/bert_mrpc_model/special_tokens_map.json',
 '/content/drive/MyDrive/bert_mrpc_model/vocab.txt',
 '/content/drive/MyDrive/bert_mrpc_model/added_tokens.json',
 '/content/drive/MyDrive/bert_mrpc_model/tokenizer.json')

In [ ]:
# 1️⃣ Import Libraries
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch
from google.colab import drive


# 2️⃣ Mount Google Drive to Access Saved Model
drive.mount('/content/drive')


# 3️⃣ Load Saved Model and Tokenizer from Google Drive
model_path = "/content/drive/MyDrive/bert_mrpc_model"
model = AutoModelForSequenceClassification.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)


# 4️⃣ Provide Sentences to Test
sentence1 = "He loves playing cricket."
sentence2 = "he is a bad person."


# 5️⃣ Tokenize Sentences for Model
inputs = tokenizer(sentence1, sentence2, return_tensors="pt", truncation=True)


# 6️⃣ Run Model for Prediction
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits
    predicted_class_id = logits.argmax().item()


# 7️⃣ Interpret and Show Result
if predicted_class_id == 1:
    print("✅ These sentences are PARAPHRASES (similar meaning).")
else:
    print("❌ These sentences are NOT PARAPHRASES (different meaning).")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
❌ These sentences are NOT PARAPHRASES (different meaning).
